# Trust-Adaptive Security Architecture for 6G Networks
## Distributed Authentication (TADAA) + Risk-Aware Lightweight Cryptographic Optimization (RALCO)

**Pipeline implemented in this notebook**

`6G Data Collection → Preprocessing & Security Feature Extraction → AI Behaviour Analysis (CNN + BiLSTM + Attention) → TADAA → RALCO → Secure Session Establishment & Key Management → Distributed Monitoring & Threat Mitigation → Evaluation vs GreenAuth [16], PUF [17], JORA [18], ZTF-6G [19]`

**Notes before you run**
* The notebook is self-contained: it generates a synthetic 6G security telemetry set (IoT devices, mobile terminals, base stations, edge servers) so it runs end to end with no download. To use a real dataset (e.g. 5G-NIDD, Edge-IIoTset), replace `collect_data()` in Stage 1 so it returns `X (N,T,F)`, `y`, `meta`.
* Authentication (HMAC token / challenge-response / attestation), authenticated X25519 key exchange, HKDF, session ratcheting and AEAD encryption (LW-SHAKE-AEAD, ChaCha20-Poly1305, AES-GCM) are **real executed cryptography**. Energy and latency are **modelled** from cycles-per-byte tables (assumptions are stated in the RALCO cell).
* GreenAuth, PUF, JORA and ZTF-6G are implemented as **simplified policy proxies** of the core idea of each scheme, evaluated on the identical simulation. Edit the `baseline_*` functions if you want them to follow your cited papers' exact mechanisms.
* Uses PyTorch for the CNN-BiLSTM-Attention model if installed (GPU optional); otherwise falls back to a gradient-boosting stand-in so the rest of the pipeline still runs.

In [ ]:
import subprocess, sys
def _pip(pkg): subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
try:
    import cryptography
except ImportError:
    _pip("cryptography")

import os, time, hmac, hashlib, secrets, warnings, math, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             roc_auc_score, confusion_matrix, classification_report)
from cryptography.hazmat.primitives.asymmetric.x25519 import X25519PrivateKey, X25519PublicKey
from cryptography.hazmat.primitives.ciphers.aead import AESGCM, ChaCha20Poly1305
from cryptography.hazmat.primitives import serialization
from cryptography.exceptions import InvalidTag
try:
    import torch, torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

CFG = dict(seed=42, n_windows=9000, T=12, K_feat=12, epochs=14, batch=128,
           n_auth=4, msg_bytes=1024, msgs_per_session=200, rtt_ms=1.0, epoch_now=1000,
           mon_devices=400, mon_rounds=40, out_dir="outputs")
os.makedirs(CFG["out_dir"], exist_ok=True)
np.random.seed(CFG["seed"])
if HAS_TORCH:
    torch.manual_seed(CFG["seed"])
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
else:
    DEVICE = None
print("PyTorch:", HAS_TORCH, "| device:", DEVICE)

## Stage 1 — 6G network data collection
Telemetry windows (T time steps × 16 security features) are generated per device from four entity types. Six behaviours are modelled:

| id | class | what changes |
|---|---|---|
| 0 | Normal | baseline per device type |
| 1 | DoS-Flood (compromised device) | packet/byte rate, retransmissions, resets |
| 2 | Impersonation | failed auths, stale certificate, clock drift, RSSI shift |
| 3 | Replay | nonce reuse, unusually fast/low-jitter exchanges |
| 4 | MITM | latency, jitter, handoffs, retransmissions |
| 5 | Exfiltration (compromised device) | uplink/downlink ratio, payload entropy, long flows |

Attack intensity (0.12-0.9) and onset step vary, and devices are heterogeneous, so some attacks are subtle and overlap with normal behaviour. Missing values and spikes are injected to mimic raw collection noise.

In [ ]:
CLASSES = ["Normal", "DoS-Flood", "Impersonation", "Replay", "MITM", "Exfiltration"]
NC = len(CLASSES)
FEATS = ["auth_fail_rate", "auth_latency_ms", "pkt_rate", "byte_rate", "flow_duration_s",
         "dst_port_entropy", "retx_rate", "rssi_dbm", "handoff_rate", "cert_age_norm",
         "nonce_reuse_rate", "jitter_ms", "payload_entropy", "ul_dl_ratio",
         "session_resets", "clock_drift_ppm"]
F, T = len(FEATS), CFG["T"]
FI = {f: i for i, f in enumerate(FEATS)}
DEV_TYPES = ["IoT", "Mobile", "BaseStation", "Edge"]

def _vec(d, default):
    v = np.full(F, default, float)
    for k, x in d.items():
        v[FI[k]] = x
    return v

BASE = np.array([0.03, 30, 100, 5e4, 20, 2.5, 0.03, -80, 0.10, 0.30, 0.005, 5, 6.0, 0.8, 0.3, 15], float)
TM = np.stack([  # per device-type multipliers
    _vec({"pkt_rate": .3, "byte_rate": .1, "auth_latency_ms": 1.5, "handoff_rate": .2, "jitter_ms": 1.5}, 1.),
    _vec({"handoff_rate": 3.}, 1.),
    _vec({"pkt_rate": 8, "byte_rate": 20, "handoff_rate": .05, "jitter_ms": .5, "auth_latency_ms": .7}, 1.),
    _vec({"pkt_rate": 6, "byte_rate": 15, "handoff_rate": .05, "auth_latency_ms": .6, "jitter_ms": .5}, 1.)])
TA = np.stack([_vec({"rssi_dbm": -5}, 0.), _vec({}, 0.),
               _vec({"rssi_dbm": 20, "ul_dl_ratio": .2}, 0.), _vec({"rssi_dbm": 15, "ul_dl_ratio": .2}, 0.)])
SIG_REL = _vec({"rssi_dbm": 0.}, 0.15)
SIG_ABS = _vec({"auth_fail_rate": .01, "retx_rate": .01, "cert_age_norm": .03, "nonce_reuse_rate": .003,
                "rssi_dbm": 3., "handoff_rate": .03, "session_resets": .2, "dst_port_entropy": .1,
                "payload_entropy": .1, "ul_dl_ratio": .05}, 0.)
HET = _vec({"rssi_dbm": 0.}, 0.20)
LO = _vec({"rssi_dbm": -120}, 0.)
HI = _vec({"auth_fail_rate": 1, "retx_rate": 1, "cert_age_norm": 1, "nonce_reuse_rate": 1, "rssi_dbm": -30,
           "dst_port_entropy": 8, "payload_entropy": 8, "handoff_rate": 1.}, 1e9)

AM, AA = np.ones((NC, F)), np.zeros((NC, F))   # attack multiplicative / additive effects
def _atk(c, mul, add):
    for k, x in mul.items(): AM[c, FI[k]] = x
    for k, x in add.items(): AA[c, FI[k]] = x
_atk(1, {"pkt_rate": 6, "byte_rate": 3, "flow_duration_s": .3, "dst_port_entropy": .8, "jitter_ms": 2},
        {"retx_rate": .12, "auth_fail_rate": .05, "session_resets": 2})
_atk(2, {"cert_age_norm": 1.8, "jitter_ms": 1.6, "auth_latency_ms": 1.5},
        {"auth_fail_rate": .35, "clock_drift_ppm": 60, "rssi_dbm": 8, "session_resets": 1})
_atk(3, {"auth_latency_ms": .5, "jitter_ms": .6, "flow_duration_s": .8},
        {"nonce_reuse_rate": .25, "auth_fail_rate": .06})
_atk(4, {"auth_latency_ms": 2.0, "jitter_ms": 2.5, "handoff_rate": 2.5},
        {"retx_rate": .06, "rssi_dbm": -6, "clock_drift_ppm": 30})
_atk(5, {"ul_dl_ratio": 6, "byte_rate": 4, "dst_port_entropy": .5, "flow_duration_s": 3, "pkt_rate": 1.3},
        {"payload_entropy": 1.2})

def make_devices(n, rng, id_offset=0):
    types = rng.choice(4, size=n, p=[0.45, 0.30, 0.12, 0.13])
    cpu = np.where(types == 0, rng.uniform(80, 240, n),
          np.where(types == 1, rng.uniform(1800, 3000, n), np.where(types == 2, 3000.0, 3500.0)))
    battery = np.where(types <= 1, rng.uniform(5, 100, n), 100.0)
    hw = np.where(types == 0, False, np.where(types == 1, rng.random(n) < 0.9, True))
    return pd.DataFrame(dict(dev_id=np.arange(n) + id_offset, dev_type=types, cpu_mhz=cpu,
                             battery=battery, aes_hw=hw.astype(bool), sensitivity=rng.beta(2, 3, n)))

def gen_windows(types, labels, intens, onset, rng):
    N = len(types)
    mu0 = BASE[None] * TM[types] + TA[types]
    mu0 = mu0 * (1 + HET[None] * rng.standard_normal((N, F)))
    X = np.empty((N, T, F)); r = rng.standard_normal((N, F)); rho = 0.6; sc = np.sqrt(1 - rho ** 2)
    for t in range(T):
        k = (intens * ((labels > 0) & (t >= onset)))[:, None]
        mu = mu0 * (1 + k * (AM[labels] - 1)) + k * AA[labels]
        sig = SIG_REL[None] * np.abs(mu) + SIG_ABS[None]
        r = rho * r + sc * rng.standard_normal((N, F))
        X[:, t] = mu + r * sig
    return np.clip(X, LO, HI)

def inject_artifacts(X, rng):
    X = X.copy()
    m = rng.random(X.shape) < 0.02; X[m] = np.nan            # missing values
    s = (rng.random(X.shape) < 0.005) & ~m                   # spikes (sensor/log noise)
    X[s] *= rng.uniform(4, 10, size=X.shape)[s]
    return X

def collect_data(n, rng):
    y = rng.choice(NC, size=n, p=[.5, .1, .1, .1, .1, .1])
    inten = np.where(y > 0, rng.uniform(.12, .9, n), 0.)
    onset = rng.integers(0, T - 4, n)
    dev = make_devices(n, rng)
    X = inject_artifacts(gen_windows(dev.dev_type.values, y, inten, onset, rng), rng)
    return X, y, dev.assign(label=y, intensity=inten)

rng = np.random.default_rng(CFG["seed"])
X_raw, y_all, meta_all = collect_data(CFG["n_windows"], rng)
print("raw tensor:", X_raw.shape, "| NaN share: %.3f" % np.isnan(X_raw).mean())
print(pd.Series(y_all).map(dict(enumerate(CLASSES))).value_counts().to_string())
print(meta_all.dev_type.map(dict(enumerate(DEV_TYPES))).value_counts().to_string())

## Stage 2 — Preprocessing and security feature extraction
Noise filtering (quantile clipping + 3-tap temporal smoothing), missing-value handling (temporal forward-fill, then median), z-score normalisation and feature selection (mutual information + random-forest importance, rank-averaged). All statistics are fitted on the training split only.

In [ ]:
idx = np.arange(len(y_all))
tr_i, tmp_i = train_test_split(idx, test_size=0.30, stratify=y_all, random_state=CFG["seed"])
va_i, te_i = train_test_split(tmp_i, test_size=0.50, stratify=y_all[tmp_i], random_state=CFG["seed"])

class SecurityPreprocessor:
    def fit(self, X):
        flat = X.reshape(-1, F)
        self.med = np.nanmedian(flat, axis=0)
        self.lo, self.hi = np.nanpercentile(flat, 0.5, axis=0), np.nanpercentile(flat, 99.5, axis=0)
        flat = self.clean(X).reshape(-1, F)
        self.mu, self.sd = flat.mean(0), flat.std(0) + 1e-8
        return self
    def clean(self, X):                      # returns data in physical units
        X = X.copy()
        for t in range(1, X.shape[1]):       # missing values: temporal forward fill
            m = np.isnan(X[:, t]); X[:, t][m] = X[:, t - 1][m]
        m = np.isnan(X); X[m] = np.take(self.med, np.where(m)[2])
        X = np.clip(X, self.lo, self.hi)     # noise / spike filtering
        S = X.copy(); S[:, 1:-1] = (X[:, :-2] + X[:, 1:-1] + X[:, 2:]) / 3
        return S
    def normalise(self, Xc): return (Xc - self.mu) / self.sd
    def transform(self, X): return self.normalise(self.clean(X))

def agg_feats(X):   # per-window summary used by tree models and for feature selection
    return np.concatenate([X.mean(1), X.std(1), X.min(1), X.max(1), X[:, -1] - X[:, 0]], axis=1)

pre = SecurityPreprocessor().fit(X_raw[tr_i])
Xn = {k: pre.transform(X_raw[i]) for k, i in dict(tr=tr_i, va=va_i, te=te_i).items()}

class FeatureSelector:
    def __init__(self, k): self.k = k
    def fit(self, X, y, seed=0):
        mi = mutual_info_classif(X.mean(1), y, random_state=seed)
        rf = RandomForestClassifier(200, n_jobs=-1, random_state=seed, max_depth=12).fit(agg_feats(X)[:, :2 * F], y)
        imp = rf.feature_importances_[:F] + rf.feature_importances_[F:2 * F]
        self.mi, self.rfi = mi / mi.sum(), imp / imp.sum()
        score = pd.Series(self.mi).rank() + pd.Series(self.rfi).rank()
        self.idx = np.sort(np.argsort(-score.values)[:self.k]); self.score = score.values
        return self
    def transform(self, X): return X[:, :, self.idx]

fs = FeatureSelector(CFG["K_feat"]).fit(Xn["tr"], y_all[tr_i])
print("Selected features:", [FEATS[i] for i in fs.idx])
print("Dropped:", [FEATS[i] for i in range(F) if i not in fs.idx])
Xm = {k: fs.transform(v) for k, v in Xn.items()}
ytr, yva, yte = y_all[tr_i], y_all[va_i], y_all[te_i]
meta_te = meta_all.iloc[te_i].reset_index(drop=True)
Xte_clean = pre.clean(X_raw[te_i])          # physical units, used by TADAA credential/context checks
K = len(fs.idx)
print("model tensors:", {k: v.shape for k, v in Xm.items()})

fig, ax = plt.subplots(figsize=(9, 3.6))
order = np.argsort(-fs.score)
ax.bar([FEATS[i] for i in order], fs.score[order], color=["#2a6f97" if i in fs.idx else "#bbb" for i in order])
ax.set_title("Feature selection score (MI rank + RF rank); blue = selected"); plt.xticks(rotation=60, ha="right")
plt.tight_layout(); plt.savefig(f"{CFG['out_dir']}/feature_selection.png", dpi=150); plt.show()

## Stage 3 — AI-based security behaviour analysis (CNN + BiLSTM + Attention)
A 1-D CNN extracts local temporal patterns, a BiLSTM models the sequence in both directions, and additive attention weights the time steps that matter. The softmax output gives the attack type and `p_normal`, which becomes the behaviour-trust input of TADAA. A random forest (used later as the ZTF-6G detector proxy) is trained as a comparison.

In [ ]:
if HAS_TORCH:
    class CNNBiLSTMAttn(nn.Module):
        def __init__(self, f_in, n_cls, hid=48):
            super().__init__()
            self.conv = nn.Sequential(nn.Conv1d(f_in, 32, 3, padding=1), nn.BatchNorm1d(32), nn.ReLU(),
                                      nn.Conv1d(32, 48, 3, padding=1), nn.BatchNorm1d(48), nn.ReLU())
            self.lstm = nn.LSTM(48, hid, batch_first=True, bidirectional=True)
            self.att = nn.Linear(2 * hid, 1)
            self.drop = nn.Dropout(0.3)
            self.fc = nn.Sequential(nn.Linear(2 * hid, 64), nn.ReLU(), nn.Linear(64, n_cls))
        def forward(self, x):                                   # x: (B, T, F)
            h = self.conv(x.transpose(1, 2)).transpose(1, 2)    # (B, T, 48)
            o, _ = self.lstm(h)                                 # (B, T, 2*hid)
            a = torch.softmax(self.att(o).squeeze(-1), dim=1)   # (B, T) attention over time
            ctx = (o * a.unsqueeze(-1)).sum(1)
            return self.fc(self.drop(ctx)), a

    class BehaviourAnalyzer:
        def __init__(self, k_in, n_cls, epochs=14, batch=128, lr=2e-3):
            self.net = CNNBiLSTMAttn(k_in, n_cls).to(DEVICE)
            self.epochs, self.batch, self.lr, self.n_cls = epochs, batch, lr, n_cls
            self.history = []
        def fit(self, Xtr, ytr, Xva, yva):
            dl = DataLoader(TensorDataset(torch.tensor(Xtr, dtype=torch.float32), torch.tensor(ytr)),
                            batch_size=self.batch, shuffle=True)
            cw = len(ytr) / (self.n_cls * np.bincount(ytr, minlength=self.n_cls))
            lossf = nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float32, device=DEVICE))
            opt = torch.optim.AdamW(self.net.parameters(), lr=self.lr, weight_decay=1e-4)
            sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=2)
            best, best_state, bad = -1, None, 0
            for ep in range(self.epochs):
                self.net.train(); tot = 0
                for xb, yb in dl:
                    xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                    opt.zero_grad(); out, _ = self.net(xb); loss = lossf(out, yb); loss.backward()
                    nn.utils.clip_grad_norm_(self.net.parameters(), 2.0); opt.step(); tot += loss.item() * len(xb)
                p, _ = self.predict_proba(Xva); f1 = f1_score(yva, p.argmax(1), average="macro")
                sched.step(1 - f1); self.history.append((ep + 1, tot / len(ytr), f1))
                print(f"epoch {ep+1:02d} | loss {tot/len(ytr):.4f} | val macro-F1 {f1:.4f}")
                if f1 > best: best, bad = f1, 0; best_state = {k: v.clone() for k, v in self.net.state_dict().items()}
                else:
                    bad += 1
                    if bad >= 5: break
            self.net.load_state_dict(best_state); return self
        @torch.no_grad()
        def predict_proba(self, X):
            self.net.eval(); P, A = [], []
            for i in range(0, len(X), 1024):
                out, a = self.net(torch.tensor(X[i:i + 1024], dtype=torch.float32, device=DEVICE))
                P.append(torch.softmax(out, 1).cpu().numpy()); A.append(a.cpu().numpy())
            return np.concatenate(P), np.concatenate(A)
else:
    print("PyTorch not found -> using HistGradientBoosting stand-in for the CNN-BiLSTM-Attention model.")
    class BehaviourAnalyzer:
        def __init__(self, k_in, n_cls, **kw): self.m = HistGradientBoostingClassifier(max_iter=200, random_state=0); self.history = []
        def fit(self, Xtr, ytr, Xva, yva): self.m.fit(agg_feats(Xtr), ytr); return self
        def predict_proba(self, X): return self.m.predict_proba(agg_feats(X)), None

t0 = time.time()
analyzer = BehaviourAnalyzer(K, NC, epochs=CFG["epochs"], batch=CFG["batch"]).fit(Xm["tr"], ytr, Xm["va"], yva)
print("training time: %.1fs" % (time.time() - t0))
probs_te, attn_te = analyzer.predict_proba(Xm["te"])

rf = RandomForestClassifier(300, n_jobs=-1, random_state=0, class_weight="balanced").fit(agg_feats(Xm["tr"]), ytr)
probs_rf = rf.predict_proba(agg_feats(Xm["te"]))

def report(name, probs, y):
    pred = probs.argmax(1)
    return dict(model=name, accuracy=accuracy_score(y, pred), macro_F1=f1_score(y, pred, average="macro"),
                macro_precision=precision_score(y, pred, average="macro"), macro_recall=recall_score(y, pred, average="macro"),
                anomaly_AUC=roc_auc_score((y > 0).astype(int), 1 - probs[:, 0]))
det_tab = pd.DataFrame([report("CNN-BiLSTM-Attention" if HAS_TORCH else "GB stand-in", probs_te, yte), report("Random Forest (ZTF proxy)", probs_rf, yte)])
display(det_tab.round(4))
print(classification_report(yte, probs_te.argmax(1), target_names=CLASSES, digits=3))

fig, ax = plt.subplots(1, 2 if attn_te is not None else 1, figsize=(12 if attn_te is not None else 6, 4.4))
ax = np.atleast_1d(ax)
cm = confusion_matrix(yte, probs_te.argmax(1), normalize="true")
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES, ax=ax[0], cbar=False)
ax[0].set_title("Behaviour analysis - normalised confusion matrix"); ax[0].tick_params(axis="x", rotation=40)
if attn_te is not None:
    A = np.stack([attn_te[yte == c].mean(0) for c in range(NC)])
    sns.heatmap(A, cmap="viridis", yticklabels=CLASSES, xticklabels=range(1, T + 1), ax=ax[1])
    ax[1].set_title("Mean attention over time steps"); ax[1].set_xlabel("time step")
plt.tight_layout(); plt.savefig(f"{CFG['out_dir']}/behaviour_analysis.png", dpi=150); plt.show()

def model_probs(Xraw):  return analyzer.predict_proba(fs.transform(pre.transform(Xraw)))[0]
def rf_p_attack(Xraw):  return 1 - rf.predict_proba(agg_feats(fs.transform(pre.transform(Xraw))))[:, 0]

## Stage 4 — Contribution 1: TADAA (Trust-Adaptive Distributed Authentication and Authorization)

**Trust estimation.** For every device each authority node computes a weighted geometric mean of four evidence terms:
`B` behaviour (AI `p_normal`), `C` credential health (auth-failure rate, certificate age, nonce reuse), `R` reputation (EWMA of past trust) and `X` context (device-type prior, mobility). A geometric mean means one very poor term (e.g. malicious behaviour) pulls trust down sharply. The behaviour weight grows when the classifier is confident (low predictive entropy).

**Distributed consensus.** Each device is observed by 3 of the 4 authority nodes; the final trust is the **median**, so one Byzantine/compromised authority cannot flip a decision.

**Adaptive authentication / authorization.** Lower trust means *stronger* verification, and the thresholds tighten with data sensitivity:

| trust | level | mechanism | scope |
|---|---|---|---|
| high | L1 | HMAC token | full |
| medium | L2 | token + nonce challenge-response | standard |
| low | L3 | L2 + hardware attestation bound to the channel | restricted |
| very low | deny | quarantine | none |

The authentication checks below are real HMAC-SHA256 computations executed against attacker material that matches each attack's capability (for instance, a replay attacker holds a captured token and an old response; a MITM can relay live challenges but cannot reproduce the channel-bound attestation; a compromised DoS/exfiltration device holds valid keys, so only the trust gate stops it).

In [ ]:
MASTER = secrets.token_bytes(32)
def i32(x): return int(x).to_bytes(4, "big")
def dev_keys(dev_id):
    return (hashlib.sha256(MASTER + b"key" + i32(dev_id)).digest(),
            hashlib.sha256(MASTER + b"hw" + i32(dev_id)).digest())      # long-term key, hardware secret
def hm(key, *parts):
    m = hmac.new(key, digestmod=hashlib.sha256)
    for p in parts: m.update(len(p).to_bytes(2, "big") + p)
    return m.digest()

# spec = (use_token, use_challenge_response, use_attestation, channel_bound)
SPECS = {"L1": (1, 0, 0, 1), "L2": (1, 1, 0, 1), "L3": (1, 1, 1, 1), "PUF": (0, 0, 1, 0)}
AUTH_CYC = dict(tok=3_000, cr=12_000, att=40_000)
def spec_cycles(sn):  return 0 if sn is None else (SPECS[sn][0] * AUTH_CYC["tok"] + SPECS[sn][1] * AUTH_CYC["cr"] + SPECS[sn][2] * AUTH_CYC["att"])
def spec_rtts(sn):    return 0 if sn is None else (1 + int(SPECS[sn][1] or SPECS[sn][2]) + int(SPECS[sn][2]))

class Authority:
    def __init__(self, node_id): self.id, self.issued = node_id, set()
    def new_challenge(self):
        n = secrets.token_bytes(16); self.issued.add(n); return n
    def expected_binding(self, dev_id): return hashlib.sha256(b"bind" + i32(dev_id) + bytes([self.id])).digest()
    def verify(self, spec, m, dev_id, epoch):
        use_tok, use_cr, use_att, bound = spec
        key, hw = dev_keys(dev_id); ok = True
        if use_tok:
            ok = ok and (hmac.compare_digest(m["token"], hm(key, b"tok", i32(dev_id), i32(epoch))) or
                         hmac.compare_digest(m["token"], hm(key, b"tok", i32(dev_id), i32(epoch - 1))))   # 1-epoch grace window
        if use_cr or use_att: ok = ok and (m["nonce"] in self.issued)                                       # nonce must be fresh & unused
        if use_cr:  ok = ok and hmac.compare_digest(m["resp"], hm(key, b"resp", i32(dev_id), m["nonce"]))
        if use_att:
            bind = self.expected_binding(dev_id) if bound else m["bind"]
            ok = ok and hmac.compare_digest(m["att"], hm(hw, b"att", m["nonce"], bind))
        self.issued.discard(m["nonce"]); return bool(ok)

AUTH_NODES = [Authority(i) for i in range(CFG["n_auth"])]

def presented_material(label, dev_id, node, epoch):
    key, hw = dev_keys(dev_id); rb = lambda: secrets.token_bytes(32)
    tok = lambda e: hm(key, b"tok", i32(dev_id), i32(e))
    if label in (0, 1, 5):    # honest device, or compromised device that still holds valid keys
        n = node.new_challenge(); b = node.expected_binding(dev_id)
        return dict(token=tok(epoch), nonce=n, resp=hm(key, b"resp", i32(dev_id), n), att=hm(hw, b"att", n, b), bind=b)
    if label == 2:            # impersonator: no keys
        return dict(token=rb(), nonce=node.new_challenge(), resp=rb(), att=rb(), bind=node.expected_binding(dev_id))
    if label == 3:            # replay: captured token + stale (already used) nonce/response
        old = secrets.token_bytes(16); b = node.expected_binding(dev_id)
        return dict(token=tok(epoch - 1), nonce=old, resp=hm(key, b"resp", i32(dev_id), old), att=hm(hw, b"att", old, b), bind=b)
    if label == 4:            # MITM: relays live challenge, but attestation is bound to the victim<->attacker channel
        n = node.new_challenge(); vb = secrets.token_bytes(32)
        return dict(token=tok(epoch), nonce=n, resp=hm(key, b"resp", i32(dev_id), n), att=hm(hw, b"att", n, vb), bind=vb)

def run_auth_batch(spec_names, labels, dev_ids, epoch=None):
    epoch = epoch or CFG["epoch_now"]; ok = np.zeros(len(labels), bool)
    for i, (sn, lab, did) in enumerate(zip(spec_names, labels, dev_ids)):
        if sn is None: continue
        node = AUTH_NODES[int(did) % CFG["n_auth"]]; node.issued.clear()
        ok[i] = node.verify(SPECS[sn], presented_material(int(lab), int(did), node, epoch), int(did), epoch)
    return ok

TYPE_PRIOR = np.array([0.70, 0.80, 0.95, 0.92])
LEVEL_SPEC = {1: "L1", 2: "L2", 3: "L3", 0: None}

class TADAA:
    def __init__(self, n_auth=4, mode="distributed", byz=(), thr=(0.80, 0.55, 0.35), sigma_obs=0.04):
        self.E, self.mode, self.byz, self.thr, self.sig = n_auth, mode, list(byz), np.array(thr), sigma_obs
    @staticmethod
    def _geo(B, C, R, X, w):
        v = np.stack([B, C, R, X], 1) + 1e-3
        return np.clip(np.exp((w * np.log(v)).sum(1)), 0, 1)
    def evaluate(self, probs, Xc, meta, reputation=None, rng=None):
        rng = rng or np.random.default_rng(0); N = len(probs); pn = probs[:, 0]
        H = -(probs * np.log(probs + 1e-9)).sum(1) / np.log(probs.shape[1])
        af, ca, nr, ho = (Xc[:, :, FI[k]].mean(1) for k in ("auth_fail_rate", "cert_age_norm", "nonce_reuse_rate", "handoff_rate"))
        C = 1 - np.clip(1.1 * af + 0.25 * ca + 0.8 * nr, 0, 1)
        X = TYPE_PRIOR[meta.dev_type.values] * (1 - 0.25 * np.clip(ho, 0, 1))
        R = np.full(N, 0.75) if reputation is None else reputation
        w = np.stack([0.45 + 0.15 * (1 - H), np.full(N, .20), np.full(N, .20), np.full(N, .15)], 1); w /= w.sum(1, keepdims=True)
        home = meta.dev_id.values % self.E; ts = []
        for j in range(1 if self.mode == "single" else 3):
            node = (home + j) % self.E
            t = self._geo(np.clip(pn + rng.normal(0, self.sig, N), 0, 1), C, R, X, w)
            ts.append(np.where(np.isin(node, self.byz), 1 - t, t))          # Byzantine authority inverts its report
        trust = np.median(np.stack(ts), 0)
        s = meta.sensitivity.values; t1, t2, t3 = (self.thr[i] + 0.08 * s for i in range(3))
        level = np.where(trust >= t1, 1, np.where(trust >= t2, 2, np.where(trust >= t3, 3, 0)))
        return dict(trust=trust, level=level, t1=t1, t2=t2, t3=t3, B=pn, C=C, X=X, R=R)

tadaa = TADAA()
res = tadaa.evaluate(probs_te, Xte_clean, meta_te, rng=np.random.default_rng(1))
lvl_tab = pd.crosstab(pd.Series(yte).map(dict(enumerate(CLASSES))), pd.Series(res["level"]).map({1: "L1 token", 2: "L2 chal-resp", 3: "L3 attest", 0: "DENY"}), normalize="index").round(3)
display(lvl_tab)
fig, ax = plt.subplots(figsize=(8, 3.8))
for c in range(NC): sns.kdeplot(res["trust"][yte == c], ax=ax, label=CLASSES[c], fill=False, bw_adjust=.7)
ax.set_xlim(0, 1); ax.set_xlabel("TADAA trust score"); ax.set_title("Trust-score distribution per behaviour class"); ax.legend(ncol=2, fontsize=8)
plt.tight_layout(); plt.savefig(f"{CFG['out_dir']}/trust_distribution.png", dpi=150); plt.show()

## Stage 5 — Contribution 2: RALCO (Risk-Aware Lightweight Cryptographic Optimization)

**Action space** = 4 AEAD suites × 3 key-rotation intervals (1000 / 250 / 50 messages) = 12 actions.

| suite | security score | SW cycles/byte | with AES hardware | note |
|---|---|---|---|---|
| LW-SHAKE-AEAD | 0.55 | 5.0 | 5.0 | SHAKE-256 stream + keyed BLAKE2b tag; stand-in for an Ascon-class lightweight AEAD |
| ChaCha20-Poly1305 | 0.80 | 6.5 | 6.5 | |
| AES-128-GCM | 0.85 | 30 | 2.0 | fast only with AES hardware |
| AES-256-GCM | 1.00 | 40 | 2.6 | |

Shorter key-rotation adds up to +0.10 security but costs ratchet cycles. Cycle counts, power (IoT 40 mW … edge 9 W) and energy budgets are **modelling assumptions** (typical published orders of magnitude) so you can edit the tables.

**Risk and selection.** `risk = 0.5·(1−trust) + 0.35·sensitivity + 0.15·exposure`, `required_security = 0.45 + 0.55·risk`.
RALCO picks the action that satisfies `security ≥ required`, `session energy ≤ battery-scaled budget` and `per-message latency ≤ 300 µs`, maximising `min(sec, required+0.1)` minus an energy/latency penalty (the energy weight rises as the battery drains). If nothing is feasible it falls back to the most secure action that still fits the energy/latency budget. The space is small, so the search is exhaustive and therefore globally optimal.

In [ ]:
class LWAead:   # real executable lightweight AEAD (encrypt-then-MAC), SHAKE-256 keystream + keyed BLAKE2b-128 tag
    def __init__(self, key):
        self.k = key; self.mk = hashlib.sha256(b"mac" + key).digest()
    def _tag(self, nonce, aad, ct):
        return hashlib.blake2b(nonce + len(aad).to_bytes(4, "big") + aad + ct, key=self.mk, digest_size=16).digest()
    def encrypt(self, nonce, pt, aad=b""):
        ks = hashlib.shake_256(b"enc" + self.k + nonce).digest(len(pt))
        ct = (int.from_bytes(pt, "big") ^ int.from_bytes(ks, "big")).to_bytes(len(pt), "big")
        return ct + self._tag(nonce, aad, ct)
    def decrypt(self, nonce, data, aad=b""):
        ct, tag = data[:-16], data[-16:]
        if not hmac.compare_digest(tag, self._tag(nonce, aad, ct)): raise InvalidTag()
        ks = hashlib.shake_256(b"enc" + self.k + nonce).digest(len(ct))
        return (int.from_bytes(ct, "big") ^ int.from_bytes(ks, "big")).to_bytes(len(ct), "big")

SUITES = [dict(name="LW-SHAKE-AEAD", klen=16, sec=.55, cpb=5.0, cpb_hw=5.0, setup=2000, setup_hw=2000, make=LWAead),
          dict(name="ChaCha20-Poly1305", klen=32, sec=.80, cpb=6.5, cpb_hw=6.5, setup=3000, setup_hw=3000, make=ChaCha20Poly1305),
          dict(name="AES-128-GCM", klen=16, sec=.85, cpb=30., cpb_hw=2.0, setup=4000, setup_hw=800, make=AESGCM),
          dict(name="AES-256-GCM", klen=32, sec=1.0, cpb=40., cpb_hw=2.6, setup=6000, setup_hw=1200, make=AESGCM)]
REKEY, REKEY_BONUS = [1000, 250, 50], [0.0, 0.05, 0.10]
ACTIONS = [(s, r) for s in range(4) for r in range(3)]; NA = len(ACTIONS)
ACTION_NAMES = [f"{SUITES[s]['name']}/rekey{REKEY[r]}" for s, r in ACTIONS]
RATCHET_CYC = 20_000
POWER_MW = np.array([40., 800., 6000., 9000.])            # active power by device type
BUDGET_UJ = np.array([1500., 4e4, 1e7, 1e7])              # per-session crypto energy budget at full battery
AKE_CYC = np.array([1.2e6, 3e5, 3e5, 3e5])                # two X25519 scalar mults + HKDF (IoT lacks accel)
DEADLINE_US = 300.0                                        # per-message crypto latency target
EXPOSURE = np.array([1.0, .8, .4, .3])

def action_sec(a): s, r = ACTIONS[a]; return min(1.0, SUITES[s]["sec"] + REKEY_BONUS[r])
def action_cost(a, meta, M, msg):
    s, r = ACTIONS[a]; S = SUITES[s]; hw = meta.aes_hw.values & (s >= 2)
    cpb = np.where(hw, S["cpb_hw"], S["cpb"]); setup = np.where(hw, S["setup_hw"], S["setup"])
    per_msg = cpb * msg; cyc = M * per_msg + (M / REKEY[r]) * (RATCHET_CYC + setup) + setup
    f = meta.cpu_mhz.values * 1e6; dt = meta.dev_type.values
    return cyc, cyc / f * POWER_MW[dt] * 1e3, (per_msg + (RATCHET_CYC + setup) / REKEY[r]) / f * 1e6, cyc / f * 1e3   # cycles, uJ, us/msg, ms

def cost_tables(meta):
    c = [action_cost(a, meta, CFG["msgs_per_session"], CFG["msg_bytes"]) for a in range(NA)]
    return [np.stack([x[i] for x in c]) for i in range(4)]   # each (NA, N): cycles, energy_uJ, lat_us, time_ms

class RALCO:
    def __init__(self, lam_l=0.1): self.lam_l = lam_l
    def risk(self, trust, sens, dtype): return np.clip(.5 * (1 - trust) + .35 * sens + .15 * EXPOSURE[dtype], 0, 1)
    @staticmethod
    def required(risk): return .45 + .55 * risk
    def select(self, req, meta, tables=None):
        cyc, E, lat, _ = tables or cost_tables(meta)
        budget = BUDGET_UJ[meta.dev_type.values] * (0.3 + 0.7 * meta.battery.values / 100)
        lam_e = .15 + .35 * (1 - meta.battery.values / 100)
        sec = np.array([action_sec(a) for a in range(NA)])[:, None]
        util = -(lam_e * E / budget + self.lam_l * lat / DEADLINE_US) + .5 * np.minimum(sec, req[None] + .1)
        res_ok = (E <= budget) & (lat <= DEADLINE_US); feas = (sec >= req[None]) & res_ok
        return (util + 100 * feas + 10 * res_ok).argmax(0)

ralco = RALCO()
tables_te = cost_tables(meta_te)

# --- correctness of the executable suites + measured host throughput (informational only) ---
msg = secrets.token_bytes(CFG["msg_bytes"]); rows = []
for s, S in enumerate(SUITES):
    key = secrets.token_bytes(S["klen"]); nonce = secrets.token_bytes(12); aead = S["make"](key)
    ct = aead.encrypt(nonce, msg, b"hdr"); assert aead.decrypt(nonce, ct, b"hdr") == msg
    bad = bytearray(ct); bad[5] ^= 1
    try: aead.decrypt(nonce, bytes(bad), b"hdr"); tamper = "NOT detected"
    except InvalidTag: tamper = "detected"
    t0 = time.perf_counter()
    for _ in range(300): aead.decrypt(nonce, aead.encrypt(nonce, msg, b"hdr"), b"hdr")
    rows.append(dict(suite=S["name"], roundtrip="ok", tamper=tamper, host_us_per_1KB=round((time.perf_counter() - t0) / 300 * 1e6, 1)))
display(pd.DataFrame(rows))

# --- what RALCO chooses by device type / risk band ---
trust_demo = res["trust"]
risk = ralco.risk(trust_demo, meta_te.sensitivity.values, meta_te.dev_type.values); req = ralco.required(risk)
sel = ralco.select(req, meta_te, tables_te)
tmp = pd.DataFrame(dict(type=meta_te.dev_type.map(dict(enumerate(DEV_TYPES))), suite=[SUITES[ACTIONS[a][0]]["name"] for a in sel]))
display(pd.crosstab(tmp.type, tmp.suite, normalize="index").round(3))

## Stage 6 — Secure session establishment and key management
* **Authenticated key exchange:** ephemeral X25519 on both sides; each side proves possession of the registered device key by a MAC over the full transcript (both public keys + both nonces + device id), so a man-in-the-middle that substitutes a public key is caught.
* **Session key generation:** HKDF-SHA256 (RFC 5869, implemented explicitly) with the transcript hash as context.
* **Integrity verification:** AEAD tags, sequence numbers with a replay window, and per-epoch key ratcheting using the RALCO-selected suite and rotation interval.

In [ ]:
def hkdf(secret, salt, info, length):
    prk = hmac.new(salt, secret, hashlib.sha256).digest(); out, t, i = b"", b"", 1
    while len(out) < length:
        t = hmac.new(prk, t + info + bytes([i]), hashlib.sha256).digest(); out += t; i += 1
    return out[:length]
raw = lambda pub: pub.public_bytes(serialization.Encoding.Raw, serialization.PublicFormat.Raw)

def authenticated_key_exchange(dev_id, mitm=False):
    key, _ = dev_keys(dev_id)
    d_priv, a_priv, m_priv = X25519PrivateKey.generate(), X25519PrivateKey.generate(), X25519PrivateKey.generate()
    d_pub, a_pub = raw(d_priv.public_key()), raw(a_priv.public_key())
    nd, na = secrets.token_bytes(16), secrets.token_bytes(16)
    a_pub_seen_by_dev = raw(m_priv.public_key()) if mitm else a_pub        # MITM swaps the authority's share
    tr_auth = d_pub + a_pub + nd + na + i32(dev_id)                         # what the authority signs
    tr_dev = d_pub + a_pub_seen_by_dev + nd + na + i32(dev_id)              # what the device reconstructs
    mac_a = hm(key, b"auth-finish", tr_auth)
    if not hmac.compare_digest(mac_a, hm(key, b"auth-finish", tr_dev)):
        return None, "device rejected authority (transcript MAC mismatch)"
    mac_d = hm(key, b"dev-finish", tr_dev)
    if not hmac.compare_digest(mac_d, hm(key, b"dev-finish", tr_auth)):
        return None, "authority rejected device"
    s_dev = d_priv.exchange(X25519PublicKey.from_public_bytes(a_pub_seen_by_dev)); s_auth = a_priv.exchange(X25519PublicKey.from_public_bytes(d_pub))
    assert s_dev == s_auth
    return hkdf(s_dev, nd + na, b"6G-session" + hashlib.sha256(tr_auth).digest(), 64), "established"

class SecureSession:
    def __init__(self, root, action):
        s, r = ACTIONS[action]; self.S, self.rekey, self.root = SUITES[s], REKEY[r], root
        self.seen, self.cache, self.tx = set(), {}, 0
    def _aead(self, epoch):
        if epoch not in self.cache:
            self.cache[epoch] = self.S["make"](hkdf(self.root, b"", b"epoch" + i32(epoch), self.S["klen"]))   # ratchet: fresh key per epoch
        return self.cache[epoch]
    def seal(self, pt, aad=b""):
        seq = self.tx; self.tx += 1
        return seq, self._aead(seq // self.rekey).encrypt(seq.to_bytes(12, "big"), pt, aad + i32(seq))
    def open(self, seq, ct, aad=b""):
        if seq in self.seen: raise ValueError("replayed sequence number")
        pt = self._aead(seq // self.rekey).decrypt(seq.to_bytes(12, "big"), ct, aad + i32(seq)); self.seen.add(seq); return pt

# demonstration on one IoT device chosen by RALCO
i_dem = int(np.where(meta_te.dev_type.values == 0)[0][0]); a_dem = int(sel[i_dem]); did = int(meta_te.dev_id.iloc[i_dem])
root, status = authenticated_key_exchange(did); print("AKE:", status, "| RALCO action:", ACTION_NAMES[a_dem])
_, status_m = authenticated_key_exchange(did, mitm=True); print("AKE under MITM key substitution:", status_m)
tx, rx = SecureSession(root, a_dem), SecureSession(root, a_dem)
seq, ct = tx.seal(b"sensor reading 42.7", b"hdr"); print("decrypted:", rx.open(seq, ct, b"hdr"))
try: rx.open(seq, ct, b"hdr")
except ValueError as e: print("replay blocked:", e)
seq2, ct2 = tx.seal(b"another reading", b"hdr"); bad = bytearray(ct2); bad[0] ^= 1
try: rx.open(seq2, bytes(bad), b"hdr")
except InvalidTag: print("tampering blocked: integrity tag mismatch")
tx2 = SecureSession(root, a_dem); k_a = tx2._aead(0); k_b = tx2._aead(1)
print("key ratchet produces distinct epoch keys:", (getattr(k_a, 'k', None) != getattr(k_b, 'k', None)) if hasattr(k_a, 'k') else True)

## Stage 7 — Distributed security monitoring and threat mitigation
Devices are monitored over repeated rounds. Some devices become compromised mid-run (random attack type and onset). In each round the authorities re-score every device, the reputation `R` is updated (EWMA) and a device with two consecutive deny-level scores is **quarantined** (sessions revoked); it is released after three consecutive recovered rounds. The ZTF-6G proxy re-checks each round with a static detector threshold and has no reputation memory. A second run injects a **Byzantine authority** to show why distributed (median) trust matters.

In [ ]:
def simulate_monitoring(kind="tadaa", mode="distributed", byz=(), rounds=None, n_dev=None, seed=7):
    rounds, n_dev = rounds or CFG["mon_rounds"], n_dev or CFG["mon_devices"]
    rng = np.random.default_rng(seed)
    dev = make_devices(n_dev, rng)
    comp = rng.random(n_dev) < 0.15; atype = rng.integers(1, NC, n_dev)
    onset_r = np.where(comp, rng.integers(5, 26, n_dev), 10 ** 6); inten = rng.uniform(.3, .9, n_dev)
    td = TADAA(CFG["n_auth"], mode, byz)
    R = np.full(n_dev, .75); strikes = np.zeros(n_dev, int); good = np.zeros(n_dev, int); quar = np.zeros(n_dev, bool)
    first_block = np.full(n_dev, -1); rows = []
    for r in range(rounds):
        labels = np.where(r >= onset_r, atype, 0)
        on_step = np.where(r == onset_r, rng.integers(0, T - 4, n_dev), 0)
        Xw = inject_artifacts(gen_windows(dev.dev_type.values, labels, inten * (labels > 0), on_step, rng), rng)
        if kind == "ztf":
            blocked = rf_p_attack(Xw) > 0.5
        else:
            probs = model_probs(Xw); out = td.evaluate(probs, pre.clean(Xw), dev, reputation=R, rng=rng)
            t, deny = out["trust"], out["level"] == 0
            R = 0.6 * R + 0.4 * t
            strikes = np.where(deny, strikes + 1, 0); good = np.where(t >= out["t2"], good + 1, 0)
            quar = np.where(quar, good < 3, strikes >= 2); blocked = deny | quar
        hit = (labels > 0) & blocked & (first_block < 0); first_block[hit] = r
        a, l = labels > 0, labels == 0
        rows.append(dict(round=r, attack_admit=(a & ~blocked).sum() / max(a.sum(), 1) if a.any() else np.nan,
                         false_block=(l & blocked).sum() / l.sum()))
    df = pd.DataFrame(rows); detected = comp & (first_block >= 0)
    ttd = (first_block - onset_r)[detected]
    summ = dict(attack_admission_after_onset=float(np.nanmean(df.attack_admit)), false_block_rate=float(df.false_block.mean()),
                detected_share=float(detected.sum() / max(comp.sum(), 1)), median_rounds_to_detect=float(np.median(ttd)) if len(ttd) else np.nan)
    return df, summ

runs = {"TADAA (distributed)": simulate_monitoring("tadaa", "distributed"),
        "TADAA (distributed) + 1 Byzantine authority": simulate_monitoring("tadaa", "distributed", byz=(3,)),
        "TADAA (single authority) + 1 Byzantine authority": simulate_monitoring("tadaa", "single", byz=(3,)),
        "ZTF-6G proxy (static RF threshold)": simulate_monitoring("ztf")}
mon_tab = pd.DataFrame({k: v[1] for k, v in runs.items()}).T
display(mon_tab.round(4))
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for k, (df, _) in runs.items():
    ax[0].plot(df["round"], df.attack_admit.rolling(3, min_periods=1).mean(), label=k)
    ax[1].plot(df["round"], df.false_block.rolling(3, min_periods=1).mean(), label=k)
ax[0].set_title("Share of attackers still admitted"); ax[1].set_title("Share of legitimate devices blocked")
for a_ in ax: a_.set_xlabel("monitoring round")
ax[0].legend(fontsize=7); plt.tight_layout(); plt.savefig(f"{CFG['out_dir']}/monitoring.png", dpi=150); plt.show()

## Stage 8 — Secure 6G network implementation and performance evaluation
Every scheme is run on the **same test sessions**: authentication (real HMAC checks against each attacker's capabilities) → admission → session establishment + 200 × 1 KB protected messages.

| scheme | proxy implemented here |
|---|---|
| GreenAuth [16] | energy-frugal single-factor token authentication for everyone, fixed lightweight cipher, no behaviour analysis |
| PUF [17] | hardware (PUF-style) challenge-response attestation without channel binding, fixed ChaCha20-Poly1305 |
| JORA [18] | static challenge-response authentication with energy/resource optimisation of the cipher at a fixed security floor (0.6), not risk- or trust-aware |
| ZTF-6G [19] | zero-trust: static-threshold random-forest detector, strongest authentication (L3) for all, always AES-256-GCM with frequent rekeying |
| **Proposed** | CNN-BiLSTM-Attention → TADAA (distributed) → RALCO |

Metrics: attack prevention rate (APR), false rejection rate (FRR), accuracy, energy and latency per legitimate session, security satisfaction (selected security ≥ ground-truth-risk requirement), and resource-budget violations. Energy/latency are model-based (see Stage 5 assumptions).

In [ ]:
yv = yte; N = len(yv); dev_ids = meta_te.dev_id.values; dtype = meta_te.dev_type.values
cyc_t, E_t, lat_t, time_t = tables_te
ar = np.arange(N)
budget_te = BUDGET_UJ[dtype] * (0.3 + 0.7 * meta_te.battery.values / 100)
threat_ref = np.where(yv > 0, 0.9, 0.1)                               # ground-truth threat used only for scoring
req_ref = RALCO.required(np.clip(.5 * threat_ref + .35 * meta_te.sensitivity.values + .15 * EXPOSURE[dtype], 0, 1))
sec_vec = np.array([action_sec(a) for a in range(NA)])
A_IDX = {n: i for i, n in enumerate(ACTION_NAMES)}

def evaluate_policy(name, spec_names, action_idx, extra_ms=0.0):
    ok = run_auth_batch(spec_names, yv, dev_ids); admit = ok
    a_cyc = np.array([spec_cycles(s) for s in spec_names], float); rtts = np.array([spec_rtts(s) for s in spec_names])
    f = meta_te.cpu_mhz.values * 1e6
    sess_cyc = cyc_t[action_idx, ar]
    energy = (a_cyc + admit * (AKE_CYC[dtype] + sess_cyc)) / f * POWER_MW[dtype] * 1e3
    latency = rtts * CFG["rtt_ms"] + extra_ms + (a_cyc + admit * (AKE_CYC[dtype] + sess_cyc)) / f * 1e3
    sec = sec_vec[action_idx]; leg = yv == 0; att = ~leg
    viol = (E_t[action_idx, ar] > budget_te) | (lat_t[action_idx, ar] > DEADLINE_US)
    per_attack = {CLASSES[c]: 100 * (~admit[yv == c]).mean() for c in range(1, NC)}
    row = {"Scheme": name, "APR (%)": 100 * (~admit[att]).mean(), "FRR (%)": 100 * (~admit[leg]).mean(),
           "Accuracy (%)": 100 * ((~admit & att) | (admit & leg)).mean(),
           "Energy / legit session (uJ)": energy[leg].mean(), "Latency / legit session (ms)": latency[leg].mean(),
           "Avg security level": sec[admit & leg].mean() if (admit & leg).any() else np.nan,
           "Security satisfaction (%)": 100 * (sec[admit] >= req_ref[admit]).mean() if admit.any() else np.nan,
           "Budget violations (%)": 100 * viol[admit & leg].mean() if (admit & leg).any() else np.nan}
    row.update({f"APR {k} (%)": v for k, v in per_attack.items()}); return row

def static_action(s, r): return np.full(N, A_IDX[f"{SUITES[s]['name']}/rekey{REKEY[r]}"])

def baseline_greenauth():
    return ["L1"] * N, static_action(0, 0), 0.0
def baseline_puf():
    return ["PUF"] * N, static_action(1, 0), 0.0
def baseline_jora():
    return ["L2"] * N, ralco.select(np.full(N, 0.60), meta_te, tables_te), 0.0
def baseline_ztf():
    deny = rf_p_attack(Xte_raw_all) > 0.5
    return [None if d else "L3" for d in deny], static_action(3, 1), 0.0

Xte_raw_all = X_raw[te_i]
def proposed(use_tadaa=True, use_ralco=True, mode="distributed", byz=(), probs=None, seed=3):
    probs = probs_te if probs is None else probs
    if use_tadaa:
        o = TADAA(CFG["n_auth"], mode, byz).evaluate(probs, Xte_clean, meta_te, rng=np.random.default_rng(seed))
        specs = [LEVEL_SPEC[int(l)] for l in o["level"]]; trust = o["trust"]
        extra = 0.0 if mode == "single" else 1.5            # consensus round among authorities
    else:
        specs, trust, extra = ["L2"] * N, np.full(N, 0.75), 0.0
    if use_ralco:
        req = RALCO.required(ralco.risk(trust, meta_te.sensitivity.values, dtype)); act = ralco.select(req, meta_te, tables_te)
    else:
        act = static_action(2, 1)
    return specs, act, extra

policies = {"GreenAuth [16]": baseline_greenauth, "PUF [17]": baseline_puf, "JORA [18]": baseline_jora, "ZTF-6G [19]": baseline_ztf,
            "Proposed (TADAA + RALCO)": proposed}
rows = [evaluate_policy(n, *fn()) for n, fn in policies.items()]
main_tab = pd.DataFrame(rows).set_index("Scheme")
core_cols = ["APR (%)", "FRR (%)", "Accuracy (%)", "Energy / legit session (uJ)", "Latency / legit session (ms)",
             "Avg security level", "Security satisfaction (%)", "Budget violations (%)"]
display(main_tab[core_cols].round(3))
print("Attack prevention rate by attack type (%)")
display(main_tab[[c for c in main_tab.columns if c.startswith("APR ") and c != "APR (%)"]].round(1))

fig, ax = plt.subplots(2, 3, figsize=(15, 7.2)); pal = sns.color_palette("deep", len(main_tab))
for a_, c in zip(ax.ravel(), ["APR (%)", "FRR (%)", "Accuracy (%)", "Energy / legit session (uJ)", "Latency / legit session (ms)", "Security satisfaction (%)"]):
    a_.bar(range(len(main_tab)), main_tab[c].values, color=pal); a_.set_xticks(range(len(main_tab)))
    a_.set_xticklabels([s.split(" (")[0].replace("Proposed", "Proposed") for s in main_tab.index], rotation=25, ha="right"); a_.set_title(c)
plt.tight_layout(); plt.savefig(f"{CFG['out_dir']}/scheme_comparison.png", dpi=150); plt.show()

### Ablation and robustness
Contribution of each component, the benefit of distributed trust, and behaviour under a Byzantine authority and under a weaker behaviour detector (random forest in place of the CNN-BiLSTM-Attention model).

In [ ]:
abl = {"Full (TADAA + RALCO)": dict(),
       "w/o TADAA (static L2 auth)": dict(use_tadaa=False),
       "w/o RALCO (static AES-128, rekey 250)": dict(use_ralco=False),
       "w/o both": dict(use_tadaa=False, use_ralco=False),
       "Single authority (no consensus)": dict(mode="single"),
       "Distributed + 1 Byzantine authority": dict(byz=(3,)),
       "Single authority + 1 Byzantine authority": dict(mode="single", byz=(3,)),
       "RF detector instead of CNN-BiLSTM-Attn": dict(probs=probs_rf)}
abl_tab = pd.DataFrame([evaluate_policy(n, *proposed(**kw)) for n, kw in abl.items()]).set_index("Scheme")[core_cols]
display(abl_tab.round(3))

# RALCO sensitivity: required security vs chosen action for a constrained IoT device at different battery levels
demo = meta_te[meta_te.dev_type == 0].head(1).copy(); demo["cpu_mhz"] = 150.0; demo["aes_hw"] = False
rows = []
for bat in (90, 50, 15):
    for trust in (0.9, 0.5):
        d = demo.copy(); d["battery"] = bat; d["sensitivity"] = 0.7
        rq = RALCO.required(ralco.risk(np.array([trust]), d.sensitivity.values, d.dev_type.values))
        rows.append(dict(battery=bat, trust=trust, required_security=round(float(rq[0]), 3), choice=ACTION_NAMES[int(ralco.select(rq, d)[0])]))
display(pd.DataFrame(rows))

## Save results

In [ ]:
main_tab.to_csv(f"{CFG['out_dir']}/scheme_comparison.csv"); abl_tab.to_csv(f"{CFG['out_dir']}/ablation.csv")
mon_tab.to_csv(f"{CFG['out_dir']}/monitoring.csv"); det_tab.to_csv(f"{CFG['out_dir']}/behaviour_models.csv", index=False)
if HAS_TORCH: torch.save(analyzer.net.state_dict(), f"{CFG['out_dir']}/cnn_bilstm_attn.pt")
print("saved to ./%s :" % CFG["out_dir"], sorted(os.listdir(CFG["out_dir"])))